In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm

# Load and filter DRKG

In [2]:
drkg_triples = pd.read_csv("/kaggle/input/datasets/rawanmahmoudrm/drkg-full-data/drkg.tsv", sep="\t",
                           header=None, names=["head", "relation", "tail"])
print("Total DRKG rows:", len(drkg_triples))
print("Number of duplicated rows:", drkg_triples.duplicated().sum())

allowed_prefixes = ["Compound::", "Side Effect::", "Atc::",
                    "Pharmacologic Class::", "Disease::", "Gene::"]

mask = (drkg_triples["head"].str.startswith(tuple(allowed_prefixes)) &
        drkg_triples["tail"].str.startswith(tuple(allowed_prefixes)))
sub_drkg = drkg_triples[mask].copy()          # .copy() avoids SettingWithCopyWarning


def extract_unprefixed_id(entity):
    for prefix in allowed_prefixes:
        if entity.startswith(prefix):
            return entity[len(prefix):]
    return None


sub_drkg["head_unprefixed"] = sub_drkg["head"].apply(extract_unprefixed_id)
sub_drkg["tail_unprefixed"] = sub_drkg["tail"].apply(extract_unprefixed_id)

drug_info = pd.read_csv("/kaggle/input/datasets/rawanmahmoudrm/updated-file/updated_file1.csv")
drug_ids = set(drug_info["id"].tolist())      # set -> fast lookups

sub_drkg = sub_drkg[sub_drkg["head_unprefixed"].isin(drug_ids) |
                    sub_drkg["tail_unprefixed"].isin(drug_ids)]

excluded_relations = {"DRUGBANK::ddi-interactor-in::Compound:Compound",
                      "Hetionet::CrC::Compound:Compound"}
sub_drkg = sub_drkg[~sub_drkg["relation"].isin(excluded_relations)]
sub_drkg = sub_drkg.drop(columns=["head_unprefixed", "tail_unprefixed"])
print(f"Number of rows: {len(sub_drkg)}")

Total DRKG rows: 5874261
Number of duplicated rows: 3
Number of rows: 134150


# Index mappings

In [3]:
entities = pd.concat([sub_drkg["head"], sub_drkg["tail"]]).unique()
entity_to_id = {e: i for i, e in enumerate(entities)}
id_to_entity = {i: e for e, i in entity_to_id.items()}

relations = sub_drkg["relation"].unique()
relation_to_id = {r: i for i, r in enumerate(relations)}

print(f"Number of unique entities: {len(entity_to_id)}")
print(f"Number of unique relations: {len(relation_to_id)}")

# Vectorised mapping (the old lambda referenced an undefined `row`)
triples_tensor = torch.LongTensor(np.stack([
    sub_drkg["head"].map(entity_to_id).values,
    sub_drkg["relation"].map(relation_to_id).values,
    sub_drkg["tail"].map(entity_to_id).values,
], axis=1))
print("Sample mapped triple:", triples_tensor[0].tolist())

Number of unique entities: 16851
Number of unique relations: 47
Sample mapped triple: [0, 0, 1945]


# ComplEx model

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

complex_dim = 256     
learning_rate = 0.005
epochs = 50
batch_size = 1024
num_neg = 5        
reg_weight = 1e-3    

num_entities = len(entity_to_id)
num_relations = len(relation_to_id)


class ComplEx(nn.Module):
    def __init__(self, n_ent, n_rel, dim):
        super().__init__()
        self.ent_re = nn.Embedding(n_ent, dim)
        self.ent_im = nn.Embedding(n_ent, dim)
        self.rel_re = nn.Embedding(n_rel, dim)
        self.rel_im = nn.Embedding(n_rel, dim)
        for emb in (self.ent_re, self.ent_im, self.rel_re, self.rel_im):
            nn.init.xavier_uniform_(emb.weight)

    def score(self, h, r, t):
        h_re, h_im = self.ent_re(h), self.ent_im(h)
        r_re, r_im = self.rel_re(r), self.rel_im(r)
        t_re, t_im = self.ent_re(t), self.ent_im(t)
        return (h_re * r_re * t_re
                + h_im * r_re * t_im
                + h_re * r_im * t_im
                - h_im * r_im * t_re).sum(dim=1)

    def n3_reg(self, h, r, t):
        reg = 0.0
        for x in (self.ent_re(h), self.ent_im(h), self.rel_re(r),
                  self.rel_im(r), self.ent_re(t), self.ent_im(t)):
            reg = reg + torch.sum(torch.abs(x) ** 3)
        return reg / h.shape[0]

    def entity_embedding(self, idx):
        idx = torch.as_tensor(idx, device=self.ent_re.weight.device)
        return torch.cat([self.ent_re(idx), self.ent_im(idx)], dim=-1)


model = ComplEx(num_entities, num_relations, complex_dim).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
triples_tensor = triples_tensor.to(device)

# Training (softplus / logistic loss with negative sampling)

In [5]:
print("Starting training...")
n = len(triples_tensor)
for epoch in range(epochs):
    model.train()
    triples_tensor = triples_tensor[torch.randperm(n, device=device)]
    epoch_loss = 0.0

    for i in tqdm(range(0, n, batch_size), leave=False):
        batch = triples_tensor[i:i + batch_size]
        h, r, t = batch[:, 0], batch[:, 1], batch[:, 2]

        pos = model.score(h, r, t)
        loss = F.softplus(-pos).mean()       

        for _ in range(num_neg):
            corrupt_head = torch.rand(len(batch), device=device) < 0.5
            rand_ent = torch.randint(0, num_entities, (len(batch),), device=device)
            nh = torch.where(corrupt_head, rand_ent, h)
            nt = torch.where(corrupt_head, t, rand_ent)
            neg = model.score(nh, r, nt)
            loss = loss + F.softplus(neg).mean() / num_neg

        loss = loss + reg_weight * model.n3_reg(h, r, t)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    print(f"Epoch {epoch + 1}/{epochs} | Loss: {epoch_loss:.4f}")

torch.save(model.state_dict(), "complex_model.pth")
print("Model saved!")

Starting training...


Epoch 1/50 | Loss: 134.5320


Epoch 2/50 | Loss: 60.1459


Epoch 3/50 | Loss: 39.0665


Epoch 4/50 | Loss: 30.0736


Epoch 5/50 | Loss: 25.9691


... (41 outputs left out)

Epoch 47/50 | Loss: 17.6901


Epoch 48/50 | Loss: 17.7411


Epoch 49/50 | Loss: 17.6647


Epoch 50/50 | Loss: 17.4538
Model saved!


# Map embeddings back to the drug table

In [6]:
embedding_size = complex_dim * 2
drug_embeddings = {}
model.eval()
with torch.no_grad():
    for idx, entity in id_to_entity.items():
        if entity.startswith("Compound::"):
            drug_id = entity[len("Compound::"):]
            if drug_id in drug_ids:
                drug_embeddings[drug_id] = model.entity_embedding(idx).cpu().numpy()


def get_embedding(drug_id):
    return drug_embeddings.get(drug_id, np.zeros(embedding_size))


drug_info["sub_drkg_embedding"] = drug_info["id"].apply(get_embedding)
print(drug_info[["id", "name", "sub_drkg_embedding"]].head(3))

drug_info.to_pickle("/kaggle/working/drug_info_with_embedding_drkg_complex.pkl")
drug_info.to_csv("/kaggle/working/drug_info_with_embedding_drkg_complex.csv", index=False)

        id          name                                 sub_drkg_embedding
0  DB01296   Glucosamine  [0.26501912, -0.16349761, -0.3797429, 0.252050...
1  DB09230  Azelnidipine  [-0.21675257, 0.15021272, 0.0014221957, -0.225...
2  DB05812   Abiraterone  [0.20992906, -0.13344944, -0.32274148, 0.13011...
